In [1]:
import cv2
import face_recognition
import numpy as np

# Load OpenCV's pre-trained Haar Cascade for eye detection
eye_cascade = cv2.CascadeClassifier('haarcascade_eye.xml')

# Use the webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

print("Webcam is open. Press 'q' to quit.")

# Dictionary to record known faces and face IDs
known_faces = []
face_ids = []
id_counter = 0

def match_face(face_encoding, threshold=0.5):
    global id_counter
    # Compare the new face with known faces
    matches = face_recognition.compare_faces(known_faces, face_encoding, tolerance=threshold)
    if True in matches:
        first_match_index = matches.index(True)
        return face_ids[first_match_index]
    else:
        # Append a new detected face to the dictionary
        known_faces.append(face_encoding)
        face_ids.append(id_counter)
        id_counter += 1
        return face_ids[-1]

while True:
    print(known_faces)
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to capture image.")
        break

    # Convert the image from BGR to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detect face locations and encodings in the frame
    face_locations = face_recognition.face_locations(rgb_frame)
    face_encodings = face_recognition.face_encodings(rgb_frame, face_locations)

    # If multiple faces are detected, find the largest face
    largest_face_index = -1
    max_face_area = 0
    for i, (top, right, bottom, left) in enumerate(face_locations):
        face_area = (bottom - top) * (right - left)
        if face_area > max_face_area:
            max_face_area = face_area
            largest_face_index = i

    # Process only the largest face
    if largest_face_index != -1:
        top, right, bottom, left = face_locations[largest_face_index]
        face_encoding = face_encodings[largest_face_index]
        
        # Additional detection for partially masked faces by checking for eyes
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        eyes = eye_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        eyes_in_face = [eye for eye in eyes if left < eye[0] < right and top < eye[1] < bottom]

        # Detect if the face is sufficiently visible
        if len(eyes_in_face) >= 1 or max_face_area > 5000:  # Adjust threshold if needed
            # Draw a green rectangle around the face and display the unique ID
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
            unique_face_id = match_face(face_encoding)
            cv2.putText(frame, f"ID: {unique_face_id}", (left, top - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.imshow('Face Detection with Unique ID', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Webcam is open. Press 'q' to quit.
[]
[]
[array([-0.0885323 ,  0.05704787,  0.08595181, -0.01499061, -0.02208209,
       -0.0982158 , -0.01739459, -0.11813892,  0.17234375, -0.07563423,
        0.26644713, -0.06765589, -0.19961351, -0.13016404, -0.02468389,
        0.18073878, -0.23163185, -0.09446397, -0.01095248,  0.01400709,
        0.10724628, -0.03923674,  0.01346941,  0.05126707, -0.06244927,
       -0.38379532, -0.11843336, -0.12291133,  0.03853235, -0.05418584,
       -0.07509676,  0.03401139, -0.18994854, -0.07861986, -0.00322717,
       -0.01521049, -0.00186587, -0.06085224,  0.20455992,  0.01753921,
       -0.23614475, -0.02396355,  0.03269556,  0.17403974,  0.1406284 ,
        0.08718652,  0.01886677, -0.10936579,  0.10781588, -0.11814982,
        0.01522743,  0.15445469,  0.0969533 ,  0.04857804, -0.00721805,
       -0.14241686,  0.0374038 ,  0.0242122 , -0.18658575, -0.00343042,
        0.07989643, -0.14599055, -0.05034985,  0.01336523,  0.21772029,
        0.05625131, -0